In [2]:
import pandas as pd
import numpy as np
import os

os.chdir(r"C:\Users\imagatsya\Downloads\credit-risk\credit-risk-model")
df = pd.read_csv("outputs/01_loaded_data.csv", low_memory=False)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(df.dtypes)

Rows: 2,260,701
Columns: 21
loan_amnt                 float64
term                       object
int_rate                  float64
installment               float64
grade                      object
emp_length                 object
home_ownership             object
annual_inc                float64
verification_status        object
loan_status                object
purpose                    object
addr_state                 object
dti                       float64
delinq_2yrs               float64
fico_range_low            float64
mths_since_last_delinq    float64
open_acc                  float64
pub_rec                   float64
revol_bal                 float64
revol_util                float64
total_acc                 float64
dtype: object


In [ ]:
# MISSING DATA REPORT 
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)
print(report[report['Missing %'] > 0])

                        Missing Count  Missing %
mths_since_last_delinq        1158535  51.246715
emp_length                     146940   6.499754
revol_util                       1835   0.081170
dti                              1744   0.077144
total_acc                          62   0.002743
pub_rec                            62   0.002743
open_acc                           62   0.002743
delinq_2yrs                        62   0.002743
annual_inc                         37   0.001637
revol_bal                          33   0.001460
fico_range_low                     33   0.001460
loan_amnt                          33   0.001460
addr_state                         33   0.001460
term                               33   0.001460
loan_status                        33   0.001460
verification_status                33   0.001460
home_ownership                     33   0.001460
grade                              33   0.001460
installment                        33   0.001460
int_rate            

In [ ]:


# 51% missing 
df['mths_since_last_delinq'] = df['mths_since_last_delinq'].fillna(999)
# 6.5% missing 
df['emp_length'] = df['emp_length'].fillna('Unknown')
# Small % missing 
df['revol_util'] = df['revol_util'].fillna(df['revol_util'].median())
df['dti'] = df['dti'].fillna(df['dti'].median())

# Tiny % missing — just drop those rows
df = df.dropna()

print(f"Rows after cleaning: {df.shape[0]:,}")
print(f"Missing values left: {df.isnull().sum().sum()}")

Rows after cleaning: 2,260,639
Missing values left: 0


In [5]:
# ── CREATE TARGET VARIABLE ───────────────────
# Keep only loans with known outcomes
# Charged Off = defaulted = 1
# Fully Paid = paid back = 0
# Remove "Current" — we don't know outcome yet

print("All loan statuses in data:")
print(df['loan_status'].value_counts())

df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])]
df['default'] = (df['loan_status'] == 'Charged Off').astype(int)

print(f"\nLoans kept : {df.shape[0]:,}")
print(f"Default rate: {df['default'].mean():.1%}")

All loan statuses in data:
loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1962
Does not meet the credit policy. Status:Charged Off        758
Default                                                     40
Name: count, dtype: int64

Loans kept : 1,345,310
Default rate: 20.0%


In [6]:
# term: " 36 months" → 36
df['term'] = df['term'].str.replace(' months','').str.strip().astype(int)

# emp_length: "10+ years" → 10, "Unknown" → -1
def clean_emp_length(x):
    if x == 'Unknown': return -1
    x = str(x).replace('years','').replace('year','')
    x = x.replace('+','').replace('<','').strip()
    try: return float(x)
    except: return -1

df['emp_length'] = df['emp_length'].apply(clean_emp_length)

print(df[['term','emp_length']].value_counts().head(10))

term  emp_length
36     10.0         320457
       1.0          154059
60     10.0         121742
36     2.0           94906
       3.0           83766
      -1.0           66200
       5.0           64228
       4.0           62176
       6.0           47459
       8.0           45337
Name: count, dtype: int64


In [7]:
# Save cleaned data 
df.to_csv("outputs/02_cleaned_data.csv", index=False)

print(f"Rows saved: {df.shape[0]:,}")
print(f"Columns   : {df.shape[1]}")
print("✓ Saved to outputs/02_cleaned_data.csv")

Rows saved: 1,345,310
Columns   : 22
✓ Saved to outputs/02_cleaned_data.csv
